# Análisis del comportamiento de una API REST bajo distintos escenarios de despliegue
**Curso:** Principios y prácticas de Desarrollo de Software Orientado a Objetos
**Autor:** _(completar)_  **Fecha:** _(completar)_

## 1. Introducción y objetivos
Se despliega la misma aplicación en: (a) Docker Compose en una sola máquina y (b) Kubernetes (K3s) con 1, 2 y 3 nodos, variando el número de réplicas (1, 2, 3). Con JMeter se genera carga con distintos números de usuarios concurrentes y se miden **tiempo medio de respuesta** y **throughput**. El objetivo es analizar el efecto del entorno, de las réplicas y de los nodos sobre la escalabilidad horizontal.

## 2. Descripción de la aplicación
API REST de catálogo de productos (FastAPI + PostgreSQL), código en `app/main.py`.

| Endpoint | Tipo | Descripción |
|---|---|---|
| `GET /products/{id}` | lectura | por clave primaria |
| `GET /products?category=&limit=` | lectura | lista filtrada (índice por categoría) |
| `GET /search?q=` | lectura | `ILIKE` sobre 20.000 filas (costosa) |
| `GET /stats` | lectura | agregación `GROUP BY` completa |
| `POST /products` | escritura | inserción |
| `PUT /products/{id}/stock` | escritura | actualización |

Base de datos: tabla `products` con **20.000 registros** (`db/init.sql`). Un solo worker de uvicorn por contenedor, de modo que el escalado se logra con réplicas.

## 3. Configuración de despliegue
- **Docker Compose:** `docker-compose.yml` (servicios `db` y `api`), en una máquina distinta a la de JMeter.
- **Kubernetes:** `k8s/*.yaml` (Namespace, Deployment+Service NodePort de la API, `topologySpreadConstraints` para repartir réplicas entre nodos y un Service `db` apuntando a la base de datos externa). **PostgreSQL corre en Docker en la VM A, fuera del clúster** (`docker-compose.db.yml`): al intentar ejecutarlo como pod en Ubuntu 26.04 (kernel 7.0), AppArmor bloqueaba las señales entre procesos de PostgreSQL y la base entraba en bucle de recuperación (manifiesto en `k8s/alternativa/`). Así, la base de datos queda fija en todos los escenarios y solo varían las réplicas y los nodos de la API. Recursos de la API fijos: requests 250m CPU / 128Mi, limits 500m CPU / 256Mi.
- **Clústeres:** K3s con 1, 2 y 3 nodos (`docs/CLUSTER_SETUP.md`). Réplicas con `kubectl scale deploy/api --replicas=N`.
- _(Completar: especificaciones de las máquinas — vCPU, RAM, proveedor, versión de K3s/Docker.)_

## 4. Metodología de carga con JMeter
- Plan `jmeter/plan.jmx`: cada usuario virtual ejecuta en bucle 6 peticiones (4 lecturas, 1 POST, 1 PUT) con IDs aleatorios.
- Niveles de carga: 10, 50, 100 y 200 usuarios concurrentes; ramp-up 10 s; duración 90 s por corrida; 3 repeticiones; 15 s de enfriamiento.
- JMeter en modo no-GUI (`jmeter/run_experiment.sh`) desde una máquina distinta a la del servidor.
- Se descartan los primeros 20 s de cada corrida (calentamiento y ramp-up) y se calculan: tiempo medio de respuesta (ms), percentil 95, throughput (req/s) y porcentaje de errores.

In [ ]:
import glob, os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
RESULTS_DIR = "../results"      # carpeta con los .jtl (en Colab: súbelos y ajusta la ruta)
WARMUP_S = 20                    # segundos a descartar al inicio de cada corrida

## 5. Carga y procesamiento de los resultados
Los archivos se llaman `<escenario>_n<nodos>_r<replicas>_t<hilos>_rep<k>.jtl`. Se calculan las métricas por corrida a partir de los datos crudos de JMeter.

In [ ]:
pat = re.compile(r"(?P<scenario>[a-z0-9]+)_n(?P<nodes>\d+)_r(?P<replicas>\d+)_t(?P<threads>\d+)_rep(?P<rep>\d+)\.jtl$")

def summarize(path):
    m = pat.search(os.path.basename(path))
    if not m:
        return None
    df = pd.read_csv(path)
    t0 = df["timeStamp"].min()
    df = df[df["timeStamp"] >= t0 + WARMUP_S * 1000]
    if df.empty:
        return None
    ok = df["success"].astype(str).str.lower() == "true"
    dur_s = (df["timeStamp"].max() + df["elapsed"].iloc[-1] - df["timeStamp"].min()) / 1000
    d = m.groupdict()
    return {
        "scenario": d["scenario"], "nodes": int(d["nodes"]), "replicas": int(d["replicas"]),
        "threads": int(d["threads"]), "rep": int(d["rep"]),
        "samples": len(df),
        "avg_ms": df["elapsed"].mean(),
        "p95_ms": df["elapsed"].quantile(0.95),
        "throughput": len(df) / dur_s,
        "error_pct": 100 * (~ok).mean(),
    }

rows = [summarize(p) for p in sorted(glob.glob(f"{RESULTS_DIR}/*.jtl"))]
runs = pd.DataFrame([r for r in rows if r])
if runs.empty:
    print("No hay resultados en", RESULTS_DIR, "- ejecuta jmeter/run_experiment.sh primero.")
else:
    runs["config"] = runs.apply(lambda r: "Docker Compose" if r.scenario == "docker"
                                else f"K8s {r.nodes}n/{r.replicas}r", axis=1)
    display(runs.head())

In [ ]:
summary = (runs.groupby(["config", "scenario", "nodes", "replicas", "threads"], as_index=False)
               .agg(avg_ms=("avg_ms", "mean"), avg_ms_std=("avg_ms", "std"),
                    p95_ms=("p95_ms", "mean"), throughput=("throughput", "mean"),
                    throughput_std=("throughput", "std"), error_pct=("error_pct", "mean"),
                    n_runs=("rep", "count")))
summary = summary.sort_values(["scenario", "nodes", "replicas", "threads"])
summary.round(2)

## 6. Resultados
### 6.1 Tiempo medio de respuesta y throughput por escenario

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for cfg, g in summary.groupby("config"):
    g = g.sort_values("threads")
    axes[0].errorbar(g["threads"], g["avg_ms"], yerr=g["avg_ms_std"].fillna(0), marker="o", capsize=3, label=cfg)
    axes[1].errorbar(g["threads"], g["throughput"], yerr=g["throughput_std"].fillna(0), marker="o", capsize=3, label=cfg)
axes[0].set(title="Tiempo medio de respuesta vs. carga", xlabel="Usuarios concurrentes", ylabel="ms")
axes[1].set(title="Throughput vs. carga", xlabel="Usuarios concurrentes", ylabel="peticiones/s")
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

### 6.2 Efecto del número de réplicas (por número de nodos)

In [ ]:
k8s = summary[summary.scenario != "docker"]
for metric, label in [("avg_ms", "Tiempo medio de respuesta (ms)"), ("throughput", "Throughput (req/s)")]:
    g = sns.relplot(data=k8s, x="threads", y=metric, hue="replicas", col="nodes", kind="line",
                    marker="o", palette="viridis", height=4, aspect=1.1)
    g.set_axis_labels("Usuarios concurrentes", label)
    g.figure.suptitle(label + " por réplicas y nodos", y=1.03)
    plt.show()

### 6.3 Mapas de calor (carga máxima y carga media)

In [ ]:
for th in sorted(k8s["threads"].unique())[-2:]:
    sub = k8s[k8s.threads == th]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.heatmap(sub.pivot(index="nodes", columns="replicas", values="throughput"), annot=True, fmt=".0f", cmap="YlGn", ax=axes[0])
    axes[0].set_title(f"Throughput (req/s) — {th} usuarios")
    sns.heatmap(sub.pivot(index="nodes", columns="replicas", values="avg_ms"), annot=True, fmt=".0f", cmap="YlOrRd_r", ax=axes[1])
    axes[1].set_title(f"Tiempo medio (ms) — {th} usuarios")
    plt.tight_layout(); plt.show()

### 6.4 Escalabilidad: speed-up del throughput respecto a 1 nodo / 1 réplica

In [ ]:
base = k8s[(k8s.nodes == 1) & (k8s.replicas == 1)][["threads", "throughput"]].rename(columns={"throughput": "base"})
sp = k8s.merge(base, on="threads")
sp["speedup"] = sp["throughput"] / sp["base"]
sp["efficiency"] = sp["speedup"] / sp["replicas"]
top = sp[sp.threads == sp.threads.max()]
display(top[["nodes", "replicas", "throughput", "speedup", "efficiency", "error_pct"]].round(2))
sns.barplot(data=top, x="replicas", y="speedup", hue="nodes")
plt.axhline(1, color="gray", ls="--"); plt.title(f"Speed-up con {sp.threads.max()} usuarios"); plt.show()

### 6.5 Tasa de errores

In [ ]:
print(summary.pivot_table(index="config", columns="threads", values="error_pct").round(2))

## 7. (Opcional) Métricas internas con Prometheus/Grafana
_Si se usaron, describir aquí la instalación (p. ej. `helm install kube-prometheus-stack`) e incluir capturas o consultas de CPU/RAM por pod y por nodo para cada configuración._

## 8. Análisis comparativo
> **Completar con tus datos reales** una vez ejecutadas las pruebas. Preguntas guía:
> 1. ¿Cómo se comparan Docker Compose y K8s 1 nodo / 1 réplica? ¿Cuánto cuesta la capa de orquestación?
> 2. En 1 nodo, ¿mejora el throughput al pasar de 1 a 2 y 3 réplicas, o la CPU del nodo ya está saturada?
> 3. Al pasar a 2 y 3 nodos, ¿el speed-up se acerca al ideal? ¿Qué eficiencia (speed-up / réplicas) se obtiene?
> 4. ¿Dónde aparece el cuello de botella? Probablemente la base de datos (única instancia, `/search` y `/stats` son costosas) o el límite de CPU de los pods.
> 5. ¿A partir de qué carga aumentan los errores o el p95?
> 6. ¿Qué efecto tiene la latencia de red entre nodos hacia la base de datos?

## 9. Conclusiones
_(Completar con 4–6 conclusiones respaldadas por los números de las tablas y gráficos anteriores.)_

## 10. Recomendaciones
_(Ejemplos a evaluar según tus datos: escalar la BD o usar réplicas de lectura/caché, ajustar `resources` y `DB_POOL_MAX`, usar HPA, índices para `/search`, más nodos solo si la API es el cuello de botella.)_